# Chapter 03：Mask 与 Memory Access

**目标**：理解边界 mask 和 stride。我们只支持容易观察的一维 positive-stride slice，例如 `base[::2]`。

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name.startswith("chapter_"):
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import triton
import triton.language as tl

from common.benchmark import bench
from common.check import assert_close
from common.utils import get_device, set_seed

device = get_device()
set_seed(0)

## 1. 为什么需要 mask

当 `N` 不是 `BLOCK_SIZE` 的整数倍时，最后一个 program 会生成越界 offset。`tl.load(..., mask=..., other=...)` 为无效 lane 提供替代值，`tl.store` 的 mask 阻止越界写入。

In [ ]:
@triton.jit
def add_contiguous_kernel(x_ptr, y_ptr, output_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    offsets = tl.program_id(0) * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    x_values = tl.load(x_ptr + offsets, mask=mask, other=0.0)
    y_values = tl.load(y_ptr + offsets, mask=mask, other=0.0)
    tl.store(output_ptr + offsets, x_values + y_values, mask=mask)

def add_contiguous(x, y):
    if x.shape != y.shape or x.ndim != 1:
        raise ValueError("x and y must be matching 1D tensors")
    if not x.is_cuda or not y.is_cuda or not x.is_contiguous() or not y.is_contiguous():
        raise ValueError("add_contiguous expects contiguous CUDA tensors")
    output = torch.empty_like(x)
    if x.numel() > 0:
        add_contiguous_kernel[(triton.cdiv(x.numel(), 256),)](
            x, y, output, x.numel(), BLOCK_SIZE=256
        )
    return output

## 2. Contiguous、slice 与 stride

contiguous tensor 的相邻逻辑元素也相邻存储，stride 通常为 1。`base[::2]` 的逻辑元素间隔两个存储位置，因此 offset 不能直接当作物理地址偏移。

In [ ]:
base = torch.arange(12, device=device, dtype=torch.float32)
sliced = base[::2]
print(f"base:   contiguous={base.is_contiguous()}, stride={base.stride()}")
print(f"sliced: contiguous={sliced.is_contiguous()}, stride={sliced.stride()}")
print(sliced)

## 3. 支持 stride 的 add kernel

逻辑 offset 乘以 tensor stride，才得到相对 base pointer 的物理元素偏移。输出是新建的 contiguous tensor，所以 store 仍使用普通 offset。

In [ ]:
@triton.jit
def add_strided_kernel(x_ptr, y_ptr, output_ptr, n_elements, x_stride, y_stride, BLOCK_SIZE: tl.constexpr):
    offsets = tl.program_id(0) * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    x_values = tl.load(x_ptr + offsets * x_stride, mask=mask, other=0.0)
    y_values = tl.load(y_ptr + offsets * y_stride, mask=mask, other=0.0)
    tl.store(output_ptr + offsets, x_values + y_values, mask=mask)

def add_strided(x, y):
    if x.shape != y.shape or x.ndim != 1 or not x.is_cuda or not y.is_cuda:
        raise ValueError("x and y must be matching 1D CUDA tensors")
    if x.stride(0) <= 0 or y.stride(0) <= 0:
        raise ValueError("only positive strides are supported")
    output = torch.empty(x.shape, device=x.device, dtype=x.dtype)
    if x.numel() > 0:
        add_strided_kernel[(triton.cdiv(x.numel(), 256),)](
            x, y, output, x.numel(), x.stride(0), y.stride(0), BLOCK_SIZE=256
        )
    return output

## 4. Correctness：连续与 sliced tensor

In [ ]:
n = 500_003
x = torch.randn(n, device=device)
y = torch.randn(n, device=device)
assert_close("contiguous add", add_contiguous(x, y), x + y)

x_slice = torch.randn(n * 2, device=device)[::2]
y_slice = torch.randn(n * 3, device=device)[::3]
assert_close("strided add", add_strided(x_slice, y_slice), x_slice + y_slice)

## 5. Benchmark

In [ ]:
print(f"contiguous torch:  {bench(lambda: x + y):.3f} ms")
print(f"contiguous triton: {bench(lambda: add_contiguous(x, y)):.3f} ms")
print(f"strided torch:     {bench(lambda: x_slice + y_slice):.3f} ms")
print(f"strided triton:    {bench(lambda: add_strided(x_slice, y_slice)):.3f} ms")

## 小结与练习

mask 解决边界有效性，stride 解决逻辑索引到物理地址的映射。

**练习**：把 slice 改成 `[::4]`，打印 stride 并重新运行 correctness check。